# Fine-Tune Patisserie AI's Intent Classifier — Llama-3.2-1B (LoRA + Sequence Classification)

Please use a **free** Tesla T4 Colab GPU to run this.

This is a custom exploration project, separate from the graded Week 5 submission notebook (which fine-tunes Qwen3-1.7B generatively via LLaMA-Factory and should be run as-is, unmodified).

## The scenario

Patisserie AI's agent routes every instructor query into one of 5 intents (`find_recipe`, `scale_recipe`, `build_indent`, `check_anomaly`, `general`) before deciding what to do with it. Today that routing is keyword rules with a fallback call to a large model (`Llama-3.3-70B-Instruct` via Nebius) for ambiguous queries — accurate (86% on our golden eval set after prompt/logic fixes) but the fallback path pays full large-model latency and cost for what is fundamentally a 5-way decision.

## Why a classification head instead of generative fine-tuning

The course notebook fine-tunes Qwen3 to *generate* the label as text (with a letter-choice workaround just to get clean single-token comparisons for its baseline). That's the right teaching tool for a general "how to fine-tune any LLaMA-Factory model" lesson, but it solves a discriminative problem via generation.

Here we do the architecturally correct thing for a closed 5-way decision: load `Llama-3.2-1B` as a **sequence classifier** (`AutoModelForSequenceClassification`, a linear head on top of the backbone that outputs 5 logits directly), and LoRA-fine-tune that. No autoregressive decoding, no label-string parsing, one forward pass to a decision — which also happens to be the honest way to make the cost/latency argument for replacing the fallback call.

## What this notebook does

1. Install dependencies
2. Authenticate with Hugging Face (Llama checkpoints are gated)
3. Load the training data generated by `finetune/generate_dataset.py`
4. Load `Llama-3.2-1B` as a 5-way sequence classifier, attach LoRA
5. Establish a majority-class baseline (what "no signal at all" looks like)
6. Train with LoRA via Hugging Face `Trainer`
7. Review the loss curve
8. Merge the LoRA adapter into the base weights
9. Evaluate on the held-out validation split, AND on `evals/golden_dataset.csv` (never seen during training) — comparing against the current production pipeline's 86%

## 1. Install Dependencies

In [ ]:
!pip install -q -U transformers accelerate peft torchao datasets scikit-learn matplotlib seaborn huggingface_hub

### Check GPU environment

In [ ]:
import torch

assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Hugging Face Login (Llama checkpoints are gated)

Unlike Qwen, Meta's Llama checkpoints require requesting access on the model page and logging in with a token before you can download weights.

1. Visit `huggingface.co/meta-llama/Llama-3.2-1B`, log in, and accept the license if you haven't already (approval is usually near-instant).
2. Create a read-access token at `huggingface.co/settings/tokens`.
3. Run the cell below and paste the token when prompted.

In [ ]:
from huggingface_hub import login

login()

## 3. Load the Training Data

Upload `train.csv` and `val.csv` from `finetune/data/` (generated by `finetune/generate_dataset.py`), plus `golden_dataset.csv` from `evals/` — our real held-out eval set, never used in training.

In [ ]:
from google.colab import files

print("Upload train.csv, val.csv, and golden_dataset.csv (select all three):")
uploaded = files.upload()

In [ ]:
import pandas as pd

CATEGORIES = ["find_recipe", "scale_recipe", "build_indent", "check_anomaly", "general"]
LABEL2ID = {c: i for i, c in enumerate(CATEGORIES)}
ID2LABEL = {i: c for c, i in LABEL2ID.items()}

df_train = pd.read_csv("train.csv")
df_val = pd.read_csv("val.csv")
df_golden = pd.read_csv("golden_dataset.csv").rename(columns={"query": "text", "intent": "label"})

for df in (df_train, df_val, df_golden):
    df["label_id"] = df["label"].map(LABEL2ID)

print(f"Train: {len(df_train)} rows")
print(df_train["label"].value_counts())
print(f"\nVal: {len(df_val)} rows")
print(df_val["label"].value_counts())
print(f"\nGolden (held-out, never trained on): {len(df_golden)} rows")
print(df_golden["label"].value_counts())

## 4. Load Base Model as a Sequence Classifier + Attach LoRA

`AutoModelForSequenceClassification` adds a randomly-initialized linear head (`score`) on top of the frozen backbone. LoRA freezes the backbone and trains small adapter matrices — but the new classification head also needs to be trainable, so it's passed via `modules_to_save` (otherwise PEFT would freeze it along with everything else and training would never move the one part of the model that actually makes the 5-way decision).

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

BASE_MODEL_NAME = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=len(CATEGORIES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    modules_to_save=["score"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 5. Tokenize the Datasets

In [ ]:
from datasets import Dataset

MAX_LENGTH = 64  # queries are short instructor sentences, not documents

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

ds_train = Dataset.from_pandas(df_train[["text", "label_id"]].rename(columns={"label_id": "labels"}))
ds_val = Dataset.from_pandas(df_val[["text", "label_id"]].rename(columns={"label_id": "labels"}))

ds_train = ds_train.map(tokenize, batched=True)
ds_val = ds_val.map(tokenize, batched=True)

## 6. Baseline: Majority-Class (What "No Signal" Looks Like)

Before training, the classification head is randomly initialized — it has no notion of the 5 labels yet, so a "zero-shot" run would just be noise. The honest baseline here is: what accuracy do you get by always guessing the most common class? Any fine-tuned result needs to clear this bar by a wide margin to mean anything.

In [ ]:
from collections import Counter

majority_label = Counter(df_train["label"]).most_common(1)[0][0]
baseline_acc_val = (df_val["label"] == majority_label).mean()
baseline_acc_golden = (df_golden["label"] == majority_label).mean()

print(f"Majority class: {majority_label}")
print(f"Majority-class baseline accuracy — val split:    {baseline_acc_val:.1%}")
print(f"Majority-class baseline accuracy — golden set:    {baseline_acc_golden:.1%}")

## 7. Train with LoRA

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="./intent_classifier_lora",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=5,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 8. Review Training — Loss Curve

Loss should fall steadily and level off. A flat or oscillating curve usually means the learning rate is off, or the dataset is too small/homogeneous for the model to find a signal.

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
train_steps = [h["step"] for h in history if "loss" in h]
train_losses = [h["loss"] for h in history if "loss" in h]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_steps, train_losses, marker="o", markersize=3, color="#1565C0")
ax.set_xlabel("Step")
ax.set_ylabel("Training loss")
ax.set_title("LoRA fine-tune — training loss")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 9. Merge the LoRA Adapter

Folds the LoRA deltas back into the base weights, producing one standalone model with no adapter overhead at inference time.

In [ ]:
merged_model = trainer.model.merge_and_unload()
merged_model.eval()
merged_model.to("cuda")
print("Adapter merged.")

## 10. Evaluate — Validation Split

Precision/recall/F1 per class, plus a confusion matrix (rows = true label, columns = predicted). This split was held out during training.

In [ ]:
import torch

@torch.no_grad()
def predict_batch(texts, model, batch_size=32):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to("cuda")
        logits = model(**inputs).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())
    return [ID2LABEL[p] for p in preds]

val_preds = predict_batch(df_val["text"].tolist(), merged_model)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(classification_report(df_val["label"], val_preds, labels=CATEGORIES, digits=3))

cm = confusion_matrix(df_val["label"], val_preds, labels=CATEGORIES)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CATEGORIES, yticklabels=CATEGORIES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Validation split — confusion matrix")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 11. Evaluate — Golden Dataset (the Real Test)

`evals/golden_dataset.csv` was never seen during training or the earlier validation step — it's the hand-curated 50-row eval set that already documents specific keyword-bleed and dict-order failure modes in the current keyword+LLM production classifier. This is the number that actually matters: does the fine-tuned model handle the same hard cases better, worse, or the same as production?

**Production pipeline reference point: 43/50 (86%)** — from `scripts/eval_intent_classifier.py`, after prompt/keyword-logic fixes.

In [ ]:
golden_preds = predict_batch(df_golden["text"].tolist(), merged_model)

golden_acc = accuracy_score(df_golden["label"], golden_preds)
print(f"Fine-tuned Llama-3.2-1B on golden set: {golden_acc:.1%} ({sum(p == l for p, l in zip(golden_preds, df_golden['label']))}/{len(df_golden)})")
print(f"Current production pipeline (reference): 86% (43/50)")
print(f"Majority-class baseline: {baseline_acc_golden:.1%}")
print()
print(classification_report(df_golden["label"], golden_preds, labels=CATEGORIES, digits=3))

cm_golden = confusion_matrix(df_golden["label"], golden_preds, labels=CATEGORIES)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm_golden, annot=True, fmt="d", cmap="Blues", xticklabels=CATEGORIES, yticklabels=CATEGORIES, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Golden dataset — confusion matrix")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

# Which specific rows does it get wrong? Cross-reference against the tag column
# (happy / edge / known_failure / adversarial) to see if it's the same hard
# cases the keyword classifier struggles with, or a different failure pattern.
wrong = df_golden.assign(predicted=golden_preds)
wrong = wrong[wrong["label"] != wrong["predicted"]]
print(f"\n{len(wrong)} misclassified rows:")
print(wrong[["id", "text", "label", "predicted", "tag"]].to_string(index=False))

## 12. Save the Merged Model

In [ ]:
MERGED_DIR = "/content/intent_classifier_merged"
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Saved merged model to {MERGED_DIR}")

## Summary

| Model | Golden-set accuracy |
|---|---|
| Majority-class baseline | see cell 6 output |
| Fine-tuned Llama-3.2-1B (LoRA, this notebook) | see cell 11 output |
| Current production pipeline (keyword rules + Llama-3.3-70B fallback) | 86% (43/50) |

This is an exploration artifact on the `finetune/intent-classifier` branch — adopting it in production would mean swapping `NEBIUS_INTENT_MODEL`'s call site in `backend/app/agent/graph.py` to call this model's inference endpoint instead of the Nebius chat completion, which is a separate decision from what this notebook demonstrates.